In [ ]:
"""
09_specific_player_value.py -- Scout AI Smart Inspector

Looks up a single player and predicts their market value using whichever
model is the better fit for their situation:

  - has_transfer_history == 1 -> "full" model
    The player has a real past transfer fee, so the market-signal features
    (past fee, contract leverage) are genuine, informative data. The full
    model gives the most accurate estimate.

  - has_transfer_history == 0 -> "performance_only" model
    The player has never been transferred, so max_career_transfer_fee is
    just a placeholder 0 -- not a real signal. Feeding that into the full
    model risks a systematic downward bias. The performance-only model
    ignores market-signal features entirely and values the player purely
    on on-pitch performance, bio, and club context -- exactly what you
    want for spotting undiscovered talent (Opportunity Mode).
"""

import os
import sys
import numpy as np
import pandas as pd
import sqlalchemy
import joblib
from dotenv import load_dotenv, find_dotenv

# ==========================================
# CONFIG & SETUP
# ==========================================

# Load environment variables
load_dotenv(find_dotenv())

WONDERKID_AGE_THRESHOLD = 22
PASSPORT_TIER_1_MAX_RANK = 15
PASSPORT_TIER_2_MAX_RANK = 60

LEAGUE_WEIGHTS = {
    "Premier League": 1.5,
    "LaLiga": 1.4,
    "Serie A": 1.3,
    "Bundesliga": 1.3,
    "Ligue 1": 1.2,
    "Eredivisie": 1.15,
    "Liga Portugal": 1.15,
    "Süper Lig": 1.05,
}

MODELS_DIR = "models"

MODEL_PATHS = {
    "full": os.path.join(MODELS_DIR, "scout_model_full.pkl"),
    "performance_only": os.path.join(MODELS_DIR, "scout_model_performance_only.pkl"),
}

FULL_FEATURES = [
    'age', 'height_in_cm', 'total_appearances', 'minutes_per_match',
    'total_goals', 'total_assists', 'goals_per_90', 'assists_per_90',
    'total_yellow_cards', 'total_red_cards', 'international_caps', 'international_goals',
    'club_squad_size', 'club_avg_age', 'stadium_seats', 'foreigners_percentage',
    'contract_months_remaining', 'wonderkid_hype', 'league_coefficient',
    'has_transfer_history', 'max_career_transfer_fee', 'most_recent_transfer_fee',
    'detailed_position', 'foot', 'passport_tier',
]

PERFORMANCE_FEATURES = [
    'age', 'height_in_cm', 'total_appearances', 'minutes_per_match',
    'total_goals', 'total_assists', 'goals_per_90', 'assists_per_90',
    'total_yellow_cards', 'total_red_cards', 'international_caps', 'international_goals',
    'club_squad_size', 'club_avg_age', 'stadium_seats', 'foreigners_percentage',
    'wonderkid_hype', 'league_coefficient', 'detailed_position', 'foot', 'passport_tier',
]

# ==========================================
# SHARED FEATURE ENGINEERING
# ==========================================

def engineer_features(player: pd.DataFrame) -> pd.DataFrame:
    player = player.copy()

    player["age"] = pd.to_numeric(player["age"], errors="coerce").fillna(0)
    player["total_appearances"] = pd.to_numeric(player["total_appearances"], errors="coerce").fillna(0)
    player["international_caps"] = pd.to_numeric(player["international_caps"], errors="coerce").fillna(0)
    player["max_career_transfer_fee"] = pd.to_numeric(player["max_career_transfer_fee"], errors="coerce").fillna(0)
    player["most_recent_transfer_fee"] = pd.to_numeric(player["most_recent_transfer_fee"], errors="coerce").fillna(0)
    player["passport_power_rank"] = pd.to_numeric(player["passport_power_rank"], errors="coerce")

    player["wonderkid_hype"] = np.where(
        player["age"] <= WONDERKID_AGE_THRESHOLD,
        player["total_appearances"] + (player["international_caps"] * 3),
        0,
    )

    conditions = [
        player["passport_power_rank"].fillna(999) <= PASSPORT_TIER_1_MAX_RANK,
        (player["passport_power_rank"].fillna(999) > PASSPORT_TIER_1_MAX_RANK)
        & (player["passport_power_rank"].fillna(999) <= PASSPORT_TIER_2_MAX_RANK),
    ]
    player["passport_tier"] = np.select(conditions, ["Tier_1", "Tier_2"], default="Tier_3")

    player["league_coefficient"] = player["competition_name"].map(LEAGUE_WEIGHTS).fillna(1.0)
    player["detailed_position"] = player["sub_position"].fillna(player["position_group"]).astype(str)

    return player


# ==========================================
# MODEL LOADING (Lazy)
# ==========================================

_loaded_models = {}

def get_model(label: str):
    if label not in _loaded_models:
        model_path = MODEL_PATHS[label]
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"[ERROR] Model '{model_path}' not found. Run the training script first.")
        _loaded_models[label] = joblib.load(model_path)
    return _loaded_models[label]


def get_top_drivers(model_pipeline, n=3):
    try:
        importances = model_pipeline.named_steps["regressor"].feature_importances_
        feature_names = model_pipeline.named_steps["preprocessor"].get_feature_names_out()
        feat_imp = pd.Series(importances, index=feature_names)
        top = feat_imp.sort_values(ascending=False).head(n)
        return [
            (idx.replace("remainder__", "").replace("cat__", "").replace("_", " ").title(), val)
            for idx, val in top.items()
        ]
    except Exception as e:
        print(f"[WARNING] Could not extract feature importance: {e}")
        return []


# ==========================================
# CORE LOOKUP + PREDICTION
# ==========================================

def analyze_player(player_name: str, engine: sqlalchemy.engine.Engine):
    query = "SELECT * FROM view_scout_master WHERE player_name ILIKE %(pattern)s"
    player = pd.read_sql(query, engine, params={"pattern": f"%{player_name}%"})

    if player.empty:
        print(f"\n[!] Player '{player_name}' not found in the database.")
        return

    # If multiple players match, just take the first one and notify the user
    if len(player) > 1:
        print(f"\n[i] Found {len(player)} matches. Showing the first result: {player['player_name'].iloc[0]}")

    player = player.iloc[[0]].copy()
    player = engineer_features(player)

    has_history = bool(player["has_transfer_history"].iloc[0])
    model_label = "full" if has_history else "performance_only"
    feature_list = FULL_FEATURES if has_history else PERFORMANCE_FEATURES
    model_pipeline = get_model(model_label)

    pred_log = model_pipeline.predict(player[feature_list])
    pred_val = np.expm1(pred_log[0])
    actual_val = player["current_market_value"].iloc[0]

    c_name = player["club_name"].iloc[0] if "club_name" in player.columns else "Unknown"

    print(f"\n==========================================")
    print(f" SCOUT AI PROFILE: {player['player_name'].iloc[0]}")
    print(f"==========================================")
    print(f" Club:                 {c_name}")
    print(f" Transfer History:     {'Yes' if has_history else 'No (never transferred)'}")
    print(f" Model Used:           {model_label}")
    print(f" Actual Market Value:  €{actual_val:,.0f}")
    print(f" ScoutAI Prediction:   €{pred_val:,.0f}")
    
    gap = pred_val - actual_val
    if gap > 0:
        print(f" Gap (Opportunity):    +€{gap:,.0f} 🟩")
    else:
        print(f" Gap (Overvalued):      €{gap:,.0f} 🟥")

    drivers = get_top_drivers(model_pipeline)
    if drivers:
        print("\n--- 💡 WHY THIS VALUATION? (Top 3 Drivers) ---")
        for clean_name, val in drivers:
            print(f"  * {clean_name:<25} : {val:.1%}")

    print("==========================================\n")


# ==========================================
# EXECUTION
# ==========================================

def main():
    db_url = os.getenv('DB_URL')
    if not db_url:
        print("[ERROR] DB_URL not found. Please ensure your .env file exists and is configured correctly.")
        sys.exit(1)

    print("[SYSTEM] Connecting to the database...")
    try:
        engine = sqlalchemy.create_engine(db_url)
        # Test connection
        with engine.connect() as _:
            pass
    except Exception as e:
        print(f"[ERROR] Could not connect to the database. Details: {e}")
        sys.exit(1)

    print("\n--- ScoutAI Smart Inspector ---")
    print("Type a player's name to analyze their market value.")
    print("Type 'exit' or 'quit' to close the tool.")

    while True:
        try:
            target = input("\nEnter player name to scout (or 'exit'): ").strip()
            if not target:
                continue
            if target.lower() in ['exit', 'quit']:
                print("Exiting ScoutAI Inspector. Goodbye!")
                break
                
            analyze_player(target, engine)
        except KeyboardInterrupt:
            print("\nExiting ScoutAI Inspector. Goodbye!")
            break
        except Exception as e:
            print(f"[ERROR] An unexpected error occurred: {e}")

if __name__ == "__main__":
    main()

[SYSTEM] Connecting to the database...

--- ScoutAI Smart Inspector ---
Type a player's name to analyze their market value.
Type 'exit' or 'quit' to close the tool.

 SCOUT AI PROFILE: Mason Greenwood
 Club:                 Olympique de Marseille
 Transfer History:     Yes
 Model Used:           full
 Actual Market Value:  €55,000,000
 ScoutAI Prediction:   €52,603,220
 Gap (Overvalued):      €-2,396,780 🟥

--- 💡 WHY THIS VALUATION? (Top 3 Drivers) ---
  * Most Recent Transfer Fee  : 36.4%
  * Max Career Transfer Fee   : 9.9%
  * Contract Months Remaining : 4.9%

Exiting ScoutAI Inspector. Goodbye!
